In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 43.4 MB/s eta 0:00:00


In [ ]:
# ==========================================================
# Notebook 05 : Word2Vec + Machine Learning
# ==========================================================

import pandas as pd
import numpy as np
import joblib

import matplotlib.pyplot as plt
import seaborn as sns
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")   # Required for newer NLTK versions

from nltk.tokenize import word_tokenize

from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split

from sklearn.linear_model import LogisticRegression

from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Libraries Imported Successfully!")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Libraries Imported Successfully!


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv(
    "/content/drive/MyDrive/Customer_project/final_02_comp.csv"
)

df.head()

,Complaint,Product,Complaint_Length,Processed_Complaint
0,Received Capital One charge card offer XXXX. A...,Credit card,714,received capital one charge card offer applied...
1,I do n't know how they got my cell number. I t...,Debt collection,52,nt know got cell number told would deal onlybw...
2,I 'm a longtime member of Charter One Bank/RBS...,Credit card,692,longtime member charter one bankrbs citizen ba...
3,"After looking at my credit report, I saw a col...",Credit reporting,84,looking credit report saw collection account b...
4,I received a call from a XXXX XXXX from XXXX @...,Debt collection,140,received call ext stating owed wanted however ...


In [ ]:
label_encoder = joblib.load(
    "/content/drive/MyDrive/Customer_project/models/label_encoder.pkl"
)

In [ ]:
X = df["Processed_Complaint"]

y = label_encoder.transform(df["Product"])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
X_train_tokens = X_train.apply(word_tokenize)
X_test_tokens = X_test.apply(word_tokenize)

X_train_tokens.head()

,Processed_Complaint
24176,"[company, reporting, outstanding, debt, balanc..."
102133,"[u, bank, rearranged, transaction, causing, mu..."
103610,"[asking, speak, deceased, son, refusing, ident..."
70582,"[company, servicing, mortgage, loan, switched,..."
54994,"[reference, complaint, number, reguards, refer..."


In [ ]:
w2v_model = Word2Vec(
    sentences=X_train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1,
    epochs=10
)

print("Vocabulary Size :", len(w2v_model.wv))

Vocabulary Size : 29434


In [ ]:
joblib.dump(
    w2v_model,
    "/content/drive/MyDrive/Customer_project/models/word2vec_model.pkl"
)

['/content/drive/MyDrive/Customer_project/models/word2vec_model.pkl']

In [ ]:
w2v_model.wv.most_similar("mortgage")

[('mortage', 0.7970064282417297),
 ('mortagage', 0.7132449150085449),
 ('mtg', 0.7115423083305359),
 ('mymortgage', 0.7089497447013855),
 ('mortg', 0.6816015243530273),
 ('mortgagei', 0.6783587336540222),
 ('mortgageto', 0.6783363819122314),
 ('mortgate', 0.6756365299224854),
 ('homethe', 0.6748191118240356),
 ('loanwhen', 0.6746633648872375)]

In [ ]:
w2v_model.wv.most_similar("credit")

[('mycredit', 0.7329815030097961),
 ('negatiave', 0.7316679358482361),
 ('inqueries', 0.7187959551811218),
 ('derragatory', 0.7049729824066162),
 ('credid', 0.7009966373443604),
 ('standingsi', 0.6938386559486389),
 ('hasis', 0.6934497952461243),
 ('credir', 0.6921253204345703),
 ('currectly', 0.6889942288398743),
 ('creditalso', 0.6889762282371521)]

In [ ]:
w2v_model.wv.most_similar("loan")

[('loanwhich', 0.7029836177825928),
 ('loansi', 0.7028534412384033),
 ('unconsolidated', 0.6997581720352173),
 ('loanthe', 0.6983394622802734),
 ('studen', 0.6980280876159668),
 ('loansmy', 0.6832695603370667),
 ('guid', 0.6710298657417297),
 ('loanthat', 0.668729305267334),
 ('lender', 0.667299211025238),
 ('consigned', 0.6662243604660034)]

In [ ]:
w2v_model.wv.most_similar("account")

[('acct', 0.7893094420433044),
 ('accunt', 0.7521517872810364),
 ('believei', 0.7265059947967529),
 ('mychecking', 0.7174443006515503),
 ('checking', 0.7035898566246033),
 ('accountis', 0.6976748108863831),
 ('aftera', 0.6937954425811768),
 ('accountwith', 0.6921985745429993),
 ('echecking', 0.6921761631965637),
 ('autotransfers', 0.6897774338722229)]

In [ ]:
def sentence_vector(tokens, model):

    word_vectors = []

    for word in tokens:

        if word in model.wv:
            word_vectors.append(model.wv[word])

    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(word_vectors, axis=0)

In [ ]:
X_train_w2v = np.array(
    X_train_tokens.apply(
        lambda words: sentence_vector(words, w2v_model)
    ).tolist()
)

In [ ]:
X_test_w2v = np.array(
    X_test_tokens.apply(
        lambda words: sentence_vector(words, w2v_model)
    ).tolist()
)

In [ ]:
print(X_train_w2v.shape)
print(X_test_w2v.shape)

(90002, 100)
(22501, 100)


In [ ]:
results = []

In [ ]:
import time

def evaluate_model(model, model_name):

    start = time.time()

    model.fit(X_train_w2v, y_train)

    training_time = time.time() - start

    start = time.time()

    y_pred = model.predict(X_test_w2v)

    prediction_time = time.time() - start

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average="weighted")
    recall = recall_score(y_test, y_pred, average="weighted")
    f1 = f1_score(y_test, y_pred, average="weighted")

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Training Time": round(training_time,2),
        "Prediction Time": round(prediction_time,2)
    })

    print("="*60)
    print(model_name)
    print("="*60)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")

    return model

In [ ]:
lr_model = evaluate_model(
    LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    "Logistic Regression (Word2Vec)"
)

Logistic Regression (Word2Vec)
Accuracy : 0.8284
Precision: 0.8251
Recall   : 0.8284
F1 Score : 0.8242


In [ ]:
svm_model = evaluate_model(
    LinearSVC(
        random_state=42
    ),
    "Linear SVM (Word2Vec)"
)

Linear SVM (Word2Vec)
Accuracy : 0.8251
Precision: 0.8220
Recall   : 0.8251
F1 Score : 0.8188


In [ ]:
results_df = pd.DataFrame(results)

results_df.sort_values(
    by="Accuracy",
    ascending=False
)

,Model,Accuracy,Precision,Recall,F1 Score,Training Time,Prediction Time
0,Logistic Regression (Word2Vec),0.828363,0.825092,0.828363,0.824203,42.18,0.02
1,Linear SVM (Word2Vec),0.825074,0.822013,0.825074,0.818759,44.44,0.03
